In [13]:
import polars as pl
from pomap.core.pomap import Pomap
from pomap.pomaps.categorical_pomap import CategoricalPomap

In [14]:
class Model:

    def __init__(self, *args, **kwargs):
        pass

    def train(self, train_df: pl.DataFrame):
        # In practice, this would set the internal model state.
        pass

    def predict(self, test_df: pl.DataFrame):
        pass

    def validate(self, validate_df: pl.DataFrame):
        pass

In [15]:
class LiftedModel:
    def __init__(self, model_constructor: type[Model], pomap: Pomap):
        self._constructor = model_constructor
        self.pomap = pomap
        self.models = {}

    def fit(self, train_df: pl.DataFrame):

        for label in self.pomap._collect_labels():
            sub = self.pomap.label_to_train(train_df, label=label)
            model = self._constructor()
            model.fit(sub)

            self.models[label] = model

    def predict(self, test_df: pl.DataFrame):
        test_df = test_df.with_row_index(name='__row_index')

        subs = []
        for label in self.pomap._collect_labels():
            sub = self.pomap.label_to_test(test_df, label=label)

            model = self.models[label]

            predictions = model.predict(
                sub
                ).rename('prediction')

            sub = sub.with_columns(predictions).select('prediction', '__row_index')
            subs.append(sub)

        test_df = test_df.join(pl.concat(subs), on='__row_index', how='left').drop('__row_index')

        return test_df


### Define a model type that makes it straightforward to test the PoMap behavor


In [16]:
class PassThroughModel:

    def __init__(self, value):
        self.value = value

    def fit(self, df):
        pass


    def predict(self, df):
        return pl.Series(values=[self.value]*df.shape[0], name='predictions')

In [17]:
pmap_cat1 = CategoricalPomap(column='cat1', labels=['a', 'b'])
pmap_cat2 = CategoricalPomap(column='cat2', labels=['c', 'd'])
pmap_cat3 = CategoricalPomap(column='cat3', labels=['e', 'f'])

In [18]:
pmap_prod_12 = pmap_cat1.product(pmap_cat2)

In [19]:
model = LiftedModel(model_constructor=lambda: PassThroughModel(1), pomap=pmap_cat1)

In [20]:
example_df = pl.from_dict(
    {'cat1': ['a', 'a', 'b', 'b'],
     'cat2': ['c', 'd', 'c', 'd'],
     'cat3': ['e', 'e', 'e', 'f'],
     }
)

In [21]:
pmap_cat1.labels

[Label({"Categorical cat1: ['a', 'b']": 'a'}),
 Label({"Categorical cat1: ['a', 'b']": 'b'})]

In [22]:
model.fit(example_df)

In [24]:
model.predict(example_df)

cat1,cat2,cat3,prediction
str,str,str,i64
"""a""","""c""","""e""",1
"""a""","""d""","""e""",1
"""b""","""c""","""e""",1
"""b""","""d""","""f""",1
